In [1]:
import os
import numpy as np
import pandas as pd
import joblib

# Regression Models
from sklearn.linear_model import Ridge
# Evaluation Metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
# Setup path
BASE_PATH = 'dataset'
OUTPUT_FOLDER = os.path.join(BASE_PATH, 'output_pipeline')
ARTIFACTS_FOLDER = os.path.join(BASE_PATH, 'artifacts')
MODEL_FOLDER = os.path.join(BASE_PATH, 'model_outputs')
EDA_FOLDER = os.path.join(BASE_PATH, 'eda_outputs')
RESULTS_FOLDER = os.path.join(BASE_PATH, 'results')

os.makedirs(MODEL_FOLDER, exist_ok=True)
os.makedirs(EDA_FOLDER, exist_ok=True)
os.makedirs(RESULTS_FOLDER, exist_ok=True)

In [3]:
# Load fitur (X)
X_train = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_X_train.parquet'))
X_val = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_X_val.parquet'))
X_test = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_X_test.parquet'))

In [4]:
# Load target (y) - convert ke Series untuk sklearn
y_train = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_y_train.parquet'))
y_val = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_y_val.parquet'))
y_test = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_y_test.parquet'))

In [5]:
# Load metadata ID (untuk traceback hasil prediksi nanti)
ID_train = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_ID_train.parquet'))
ID_val = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_ID_val.parquet'))
ID_test = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_ID_test.parquet'))

In [6]:
scaler = joblib.load(os.path.join(ARTIFACTS_FOLDER, 'minmax_scaler.joblib'))
winsor = joblib.load(os.path.join(ARTIFACTS_FOLDER, 'winsor_bounds.joblib'))
feat_select = joblib.load(os.path.join(ARTIFACTS_FOLDER, 'feature_selection_info.joblib'))

/home/aliarridha/miniconda3/envs/modeling1/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.6.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [7]:
TARGET = feat_select['target']

In [8]:
y_train_arr = y_train[TARGET].values
y_val_arr = y_val[TARGET].values
y_test_arr = y_test[TARGET].values

In [9]:
print(ID_test.head())
print(ID_test.columns)

         kabupaten_kota   nama_wilayah_bersih  tahun
0       Kab. Aceh Barat       kab. aceh barat   2024
1       Kab. Aceh Barat       kab. aceh barat   2025
2  Kab. Aceh Barat Daya  kab. aceh barat daya   2024
3  Kab. Aceh Barat Daya  kab. aceh barat daya   2025
4       Kab. Aceh Besar       kab. aceh besar   2024
Index(['kabupaten_kota', 'nama_wilayah_bersih', 'tahun'], dtype='object')


In [10]:
print("Mulai training Linear Regression")

# alpha kecil dulu sebagai baseline regularization
ridge_model = Ridge(alpha=1.0)

# training
ridge_model.fit(X_train, y_train)

print("Training selesai")

Mulai training Linear Regression
Training selesai


In [11]:
# Prediksi
ridge_train_pred = ridge_model.predict(X_train)
ridge_val_pred = ridge_model.predict(X_val)
ridge_test_pred = ridge_model.predict(X_test)

In [12]:
def hitung_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(
        mean_squared_error(y_true, y_pred)
    )
    r2 = r2_score(y_true, y_pred)
    return mae, rmse, r2

In [13]:
# evaluasi train
ridge_train_mae, ridge_train_rmse, ridge_train_r2 = hitung_metrics(
    y_train,
    ridge_train_pred
)


In [14]:
# evaluasi validation
ridge_val_mae, ridge_val_rmse, ridge_val_r2 = hitung_metrics(
    y_val,
    ridge_val_pred
)

In [15]:
# evaluasi test
ridge_test_mae, ridge_test_rmse, ridge_test_r2 = hitung_metrics(
    y_test,
    ridge_test_pred
)

In [16]:
print("HASIL RIDGE REGRESSION")

print("\nTrain")
print(f"MAE  : {ridge_train_mae:.4f}")
print(f"RMSE : {ridge_train_rmse:.4f}")
print(f"R2   : {ridge_train_r2:.4f}")

print("\nValidation")
print(f"MAE  : {ridge_val_mae:.4f}")
print(f"RMSE : {ridge_val_rmse:.4f}")
print(f"R2   : {ridge_val_r2:.4f}")

print("\nTest")
print(f"MAE  : {ridge_test_mae:.4f}")
print(f"RMSE : {ridge_test_rmse:.4f}")
print(f"R2   : {ridge_test_r2:.4f}")

HASIL RIDGE REGRESSION

Train
MAE  : 4.6410
RMSE : 5.9641
R2   : 0.6913

Validation
MAE  : 4.4165
RMSE : 5.7735
R2   : 0.6852

Test
MAE  : 4.8601
RMSE : 6.2536
R2   : 0.6491


In [17]:
# Simpan Model Ridge Regression
joblib.dump(
    ridge_model,
    os.path.join(BASE_PATH, 'models', 'ridge_regression.joblib')
)
print("Model berhasil disimpan.")

Model berhasil disimpan.


In [18]:
# Simpan metrics
metrics_ridge = pd.DataFrame({
    "model": ["Ridge Regression"],
    "train_mae": [ridge_train_mae],
    "train_rmse": [ridge_train_rmse],
    "train_r2": [ridge_train_r2],
    "val_mae": [ridge_val_mae],
    "val_rmse": [ridge_val_rmse],
    "val_r2": [ridge_val_r2],
    "test_mae": [ridge_test_mae],
    "test_rmse": [ridge_test_rmse],
    "test_r2": [ridge_test_r2]
})

os.makedirs("results", exist_ok=True)

metrics_ridge.to_csv(
    os.path.join(BASE_PATH, "results", "ridge_regression_metrics.csv"),
    index=False
)
print("Metrics berhasil disimpan.")

Metrics berhasil disimpan.


In [19]:
# Simpan hasil prediksi test
hasil_pred_lr = ID_test.copy()

hasil_pred_lr["actual"] = y_test.values
hasil_pred_lr["prediction_ridge"] = ridge_test_pred

hasil_pred_lr.to_csv(
    os.path.join(BASE_PATH, "results", "ridge_regression_predictions.csv"),
    index=False
)

print("Hasil prediksi berhasil disimpan.")

# preview hasil
print("\nPreview prediction:")
print(hasil_pred_lr.head())

Hasil prediksi berhasil disimpan.

Preview prediction:
         kabupaten_kota   nama_wilayah_bersih  tahun  actual  prediction_ridge
0       Kab. Aceh Barat       kab. aceh barat   2024   54.82         49.428896
1       Kab. Aceh Barat       kab. aceh barat   2025   53.53         50.581292
2  Kab. Aceh Barat Daya  kab. aceh barat daya   2024   55.15         51.291792
3  Kab. Aceh Barat Daya  kab. aceh barat daya   2025   66.77         52.472856
4       Kab. Aceh Besar       kab. aceh besar   2024   51.82         49.722846
